In [1]:
import urllib.request
import os
from datetime import datetime

Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних

In [2]:
def download_vhi_data():
    os.makedirs("vhi_data", exist_ok=True)

    existing = os.listdir("vhi_data")
    for i in range(1, 28):
        if any(f"vhi_id_{i}_" in filename for filename in existing):
               continue
        else:
            now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            filename = f"vhi_data/vhi_id_{i}_{now}.csv"
            vhiurl = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={i}&year1=1981&year2=2024&type=Mean"

            urllib.request.urlretrieve(vhiurl, filename)

    print("all done")

In [3]:
download_vhi_data()

all done


Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області

In [4]:
import pandas as pd
import os

def cleaning_df_data():
    all_dfs = []
    headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']

    for filename in os.listdir("vhi_data"):
        filepath = f"vhi_data/{filename}"
        id_prov = int(filename.split('_')[2])
        df_temp = pd.read_csv(filepath, header=None, skiprows=2, names=headers)
        df_temp['province'] = id_prov
        df_temp = df_temp.drop('empty', axis=1)
    
        all_dfs.append(df_temp)
    
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df['Year'] = final_df['Year'].astype(str).str.replace('<tt><pre>', '', regex=False)
    final_df['Year'] = final_df['Year'].astype(str).str.replace('</pre>', '', regex=False)
    final_df['Year'] = pd.to_numeric(final_df['Year'], errors='coerce')
    final_df = final_df.dropna()
    final_df['Year'] = final_df['Year'].astype(int)
    final_df['Week'] = final_df['Week'].astype(int)
    final_df = final_df[final_df['VHI'] >= 0]
    return final_df

In [5]:
df = cleaning_df_data()
df.head()

,Year,Week,SMN,SMT,VCI,TCI,VHI,province
0,1982,1,0.059,258.24,51.11,48.78,49.95,10
1,1982,2,0.063,261.53,55.89,38.20,47.04,10
2,1982,3,0.063,263.45,57.30,32.69,44.99,10
3,1982,4,0.061,265.10,53.96,28.62,41.29,10
4,1982,5,0.058,266.42,46.87,28.57,37.72,10


Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 

In [8]:
def replace_id_province(df):
    ua_provinces = {     #словник
1: 22, # Черкаська
2: 24, # Чернігівська
3: 23, # Чернівецька
4: 25, # Крим
5: 3, # Дніпропетровська
6: 4, # Донецька
7: 8, # Івано-Франківська
8: 19, # Харківська
9: 20, # Херсонська
10: 21, # Хмельницька
11: 9, # Київська
12: 26, # м. Київ
13: 10, # Кіровоградська
14: 11, # Луганська
15: 12, # Львівська
16: 13, # Миколаївська
17: 14, # Одеська
18: 15, # Полтавська
19: 16, # Рівненська
20: 27, # Севастополь
21: 17, # Сумська
22: 18, # Тернопільська
23: 6, # Закарпатська
24: 1, # Вінницька
25: 2, # Волинська
26: 7, # Запорізька
27: 5, # Житомирська
}
    df['province']=df['province'].replace(ua_provinces)
    return df


In [9]:
repl_df = replace_id_province(df)
repl_df.head()

,Year,Week,SMN,SMT,VCI,TCI,VHI,province
0,1982,1,0.059,258.24,51.11,48.78,49.95,21
1,1982,2,0.063,261.53,55.89,38.20,47.04,21
2,1982,3,0.063,263.45,57.30,32.69,44.99,21
3,1982,4,0.061,265.10,53.96,28.62,41.29,21
4,1982,5,0.058,266.42,46.87,28.57,37.72,21


Ряд VHI для області за вказаний рік

In [10]:
def vhi_province_year(repl_df, year, id_province):
    mask = (repl_df['Year'] == year) & (repl_df['province'] == id_province)
    columns_to_show = ['province', 'Year', 'VHI']
    if 'Week' in repl_df.columns:
        columns_to_show.insert(2, 'Week')
        
    sample = repl_df[mask][columns_to_show]
    return sample

year = int(input('Введіть рік: '))
id_province = int(input('Введіть ID області: '))

result = vhi_province_year(repl_df, year, id_province)

display(result)

,province,Year,Week,VHI
32202,6,1999,1,35.16
32203,6,1999,2,38.50
32204,6,1999,3,43.96
32205,6,1999,4,48.92
32206,6,1999,5,51.29
32207,6,1999,6,53.40
32208,6,1999,7,54.47
32209,6,1999,8,54.82
32210,6,1999,9,54.11
32211,6,1999,10,51.83


Ряд VHI за вказаний діапазон років для вказаних областей

In [12]:
year_start = int(input("введіть рік початку: "))
year_end = int(input("введіть рік кінця: "))

id_provinces_input = input("введіть ID областей через пробіл (наприклад: 1 5 12): ")
id_provinces_list = [int(i) for i in id_provinces_input.split()] #усі id областей оформлюю в list, щоб передати як аргумент

def vhi_yearsrange_idrange(repl_df, year_start, year_end, id_provinces_list):
    mask = (repl_df['Year'].between(year_start, year_end)) & (repl_df['province'].isin(id_provinces_list)) #створюю окремо маску
    df_yearsrange_idrange = repl_df[mask][['province', 'Year', 'VHI']] #окремо приміняю маску до фрейму
    
    return df_yearsrange_idrange

result2 = vhi_yearsrange_idrange(repl_df, year_start, year_end, id_provinces_list)
display(result2)

,province,Year,VHI
50098,3,1999,43.83
50099,3,1999,43.90
50100,3,1999,42.91
50101,3,1999,43.60
50102,3,1999,43.38
...,...,...,...
52486,4,2001,57.35
52487,4,2001,58.01
52488,4,2001,59.09
52489,4,2001,62.65


Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани

In [ ]:
year_start = int(input("Введіть рік початку: "))
year_end = int(input("Введіть рік кінця: "))

id_provinces_input = input("введіть ID областей через пробіл (наприклад: 1 5 12): ")
id_provinces_list = [int(i) for i in id_provinces_input.split()] #усі id областей оформлюю в list, щоб передати як аргумент

def minmax_average_median_yearsrange_idrange(repl_df, year_start, year_end, id_provinces_list):
    filtered_vhi = repl_df[(repl_df['Year'].between(year_start, year_end)) & (repl_df['province'].isin(id_provinces_list))]['VHI']
    
    if filtered_vhi.empty:
        return "За вказаними критеріями даних не знайдено."
        
    stats = filtered_vhi.agg(['min', 'max', 'mean', 'median'])
    return stats

result3 = minmax_average_median_yearsrange_idrange(repl_df, year_start, year_end, id_provinces_list)
display(result3)

min       17.400000
max       80.440000
mean      47.445865
median    46.605000
Name: VHI, dtype: float64